# Tune an IMSTE embedding on an MNIST-family dataset

Choose `MNIST`, `Fashion-MNIST`, or `Kuzushiji-MNIST` in the data cell below. The default search balances 5-fold k-NN accuracy (60%) and robust spatial coverage (40%). It fits each trial for the estimator's default 200 epochs.

Install the optional optimizer with `python -m pip install 'imste[tuning]'` if needed.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

repo_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'notebooks' / 'src' / 'digits_sweep.py').is_file()
)
sys.path.insert(0, str(repo_root / 'notebooks' / 'src'))
sys.path.insert(0, str(repo_root))

from digits_sweep import (
    MNIST_FAMILY_DATASETS,
    load_mnist_per_class_data,
    load_mnist_family_data,
    make_live_tuning_display,
    plot_mnist_embedding,
)
from imste import IteratedMinimumSpanningTreeEmbedder

## Choose and prepare the data

Choose one of `MNIST_FAMILY_DATASETS` below. Sampling is deterministic and balanced; the default is 2,000 images per class. The helper uses the project cache when available and downloads the selected dataset otherwise.

In [ ]:
DATASET = 'MNIST'  # Also: 'Fashion-MNIST' or 'Kuzushiji-MNIST'
SAMPLES_PER_CLASS = 2000
RANDOM_STATE = 42

if DATASET not in MNIST_FAMILY_DATASETS:
    raise ValueError(f'DATASET must be one of {MNIST_FAMILY_DATASETS}.')

X, y, data_info = load_mnist_per_class_data(
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
    dataset_name=DATASET,
)
print(
    f"Loaded {len(X):,} images from {data_info['dataset']} ({SAMPLES_PER_CLASS} per class; {X.shape[1]} pixel features)."
)
display(data_info['class_counts'].to_frame())

## Set up the search

The `.tune()` defaults search `n_msts` (3–30), `learning_rate` (0.01–0.1), `negative_ratio` (1–10), `gravity_strength` (0.0–1.0), and `mean_shift_iterations` (0–5). They run 40 total trials, with 20 initial Latin hypercube samples, `k=5`, and coverage weight 0.4. `mapper` uses the estimator default of 200 epochs per fit.

In [ ]:
mapper = IteratedMinimumSpanningTreeEmbedder(
    random_state=RANDOM_STATE,
    device='auto',
)

## Watch intermediate results

The table reports accuracy, spatial coverage, and the weighted objective after every trial. The plot updates when a trial improves the combined objective.

In [ ]:
show_trial, trial_rows = make_live_tuning_display(
    y, figsize=(12, 8)
)

started = time.perf_counter()
search = mapper.tune(
    X, y,
    trial_callback=show_trial,
)
elapsed = time.perf_counter() - started
print(f'Search and final refit completed in {elapsed / 60:.1f} minutes')

## Best result

The estimator is refitted on all selected images using the configuration with the highest weighted objective. The returned result retains the separate accuracy, coverage, and combined scores.

In [ ]:
print(f"Best 5-fold {search.k}-NN accuracy: {search.best_accuracy:.3f}")
print(f"Best spatial coverage: {search.best_coverage:.3f}")
print(f"Best combined objective: {search.best_objective:.3f}")
display(pd.Series(search.best_params, name='best value').to_frame())
plot_mnist_embedding(
    mapper.embedding_, y,
    title=(
        f'Final refit: accuracy = {search.best_accuracy:.3f}; '        f'coverage = {search.best_coverage:.3f}'
    ),
    figsize=(12, 8),
)

## Refit on the full dataset

After tuning on the selected balanced sample, load every row from the chosen dataset and fit the selected parameters again. The live search scores above are measured on the tuning sample; this final embedding uses the full dataset.

In [ ]:
X_full, y_full, full_data_info = load_mnist_family_data(DATASET)
mapper.set_params(**search.best_params)

started = time.perf_counter()
mapper.fit(X_full)
elapsed = time.perf_counter() - started
print(
    f"Refit on {len(X_full):,} {full_data_info['dataset']} images "
    f"in {elapsed / 60:.1f} minutes"
)
display(full_data_info['class_counts'].to_frame())
plot_mnist_embedding(
    mapper.embedding_, y_full,
    title=f"Full {DATASET} embedding using the best search parameters",
    figsize=(12, 8),
)